# 12-Hour-Ahead Significant Wave Height Forecasting

This notebook compares several approaches to predicting `hsig_m` 12 hours in advance for the Mooloolaba wave buoy. All shared logic lives in `src/forecast/`; this notebook is the experimental surface.

**Problem.** Given wave buoy observations at 30-minute cadence up to time *t*, predict `hsig_m` at time *t + 12h* (i.e. 24 steps ahead).

**Why 12 hours?** It's the surf-forecast sweet spot: long enough that persistence ("it'll be the same as now") is no longer a great predictor, short enough that atmospheric chaos doesn't dominate.

**Evaluation.** Chronological 80/20 split. Metrics: MAE, RMSE, Bias, and skill score vs. persistence. A positive skill score is the honest test — any model that can't beat persistence has no predictive value.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

import forecast as fc
import viz

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Horizon: {fc.HORIZON_HOURS}h = {fc.HORIZON_STEPS} steps at {fc.SAMPLING_FREQ_MINUTES}-minute cadence")


## 1. Load and inspect

The cleaned CSV comes from `python -m wave_data`. Gaps in the buoy record appear as NaN rows on the regular 30-minute grid; we'll handle those next.

In [ ]:
df_raw = fc.load_data()
print(f"{len(df_raw):,} rows spanning {df_raw.index.min().date()} \u2192 {df_raw.index.max().date()}")
print("\nMissing values per column:")
print(df_raw.isna().sum())
df_raw.head(3)

### Gap handling

Lag/rolling features propagate NaN. Three reasonable options:

1. **Drop rows with any NaN feature** — clean, but sacrifices recent data if a single channel fails.
2. **Interpolate** — reasonable for smooth geophysical signals over short gaps.
3. **Leave NaN and let the evaluation harness mask rows** — what our `evaluate()` does by default.

Short gaps in hsig_m/hmax_m/tz_s/tp_s are well-behaved (smooth series), so we'll interpolate. SST has the most missing values (~10%) but changes slowly, so interpolation is safe there too.

In [ ]:
df = df_raw.interpolate(limit=48).bfill().ffill()
assert df.isna().sum().sum() == 0, "expected fully imputed frame"
print(f"After imputation: {len(df):,} rows, 0 NaNs")

### Visualise hsig_m and its 12h-ahead autocorrelation

How much signal is even available at this horizon?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz.plot_series(df["hsig_m"], title="hsig_m (2015–2025)", ylabel="Hs (m)", ax=axes[0])
viz.autocorrelation_curve(
    df["hsig_m"], max_hours=72, highlight_hours=[fc.HORIZON_HOURS], ax=axes[1],
)
plt.tight_layout()
plt.show()

print(f"Autocorrelation at {fc.HORIZON_HOURS}h lag: {df['hsig_m'].autocorr(lag=fc.HORIZON_STEPS):.3f}")


## 2. Feature analysis and forecast feasibility

Before fitting anything (else), a sanity check: does the data even contain 12h-ahead signal, and if so, where does it live? Two heatmaps answer that:

1. **Cross-correlation of each feature against `hsig_m` at a range of horizons** — shows which channels carry predictive signal and how quickly it decays.
2. **Autocorrelation of `hsig_m` across a lookback × forecast-horizon grid** — shows whether a longer history buys more predictive power at a given horizon.

Rough reading guide: for a (near-)linear model, |r| = 0.5 translates to ~13% RMSE reduction over the mean, |r| = 0.8 to ~40%. Anything under |r| ≈ 0.1 is effectively noise at the sample sizes we have.


In [ ]:
# 2.1 — features at t vs hsig_m at t+h
df_feats = fc.encode_circular(df)
_, xcorr = viz.feature_horizon_heatmap(
    df_feats,
    target_col="hsig_m",
    feature_cols=["hsig_m", "hmax_m", "tz_s", "tp_s", "peak_dir_deg_sin", "peak_dir_deg_cos", "sst_c"],
    source_label="mooloolaba",
)
plt.tight_layout()
plt.show()
xcorr


In [ ]:
# 2.2 — hsig_m lookback × forecast horizon
_, lb_grid = viz.lookback_horizon_heatmap(df["hsig_m"], source_label="mooloolaba")
plt.tight_layout()
plt.show()
lb_grid


### Takeaways (from the numbers above)

- **`hsig_m` self-persistence sets the ceiling.** r = 0.98 at +1h, **0.81 at +12h**, 0.46 at +48h, 0.31 at +72h. This is the curve any forecaster has to beat.
- **`hmax_m` is nearly redundant** with `hsig_m` — correlation within ~0.02 across every horizon. Physically unsurprising (max height tracks significant height), but it means there's little complementary signal to extract from it.
- **`tz_s` (zero-crossing period) has a modest standalone signal** — r = 0.27 at +12h, decaying smoothly with horizon.
- **`tp_s` (peak period) is surprisingly close to noise** — |r| < 0.03 at every horizon. The raw channel is likely too jumpy (it's a binned spectral maximum); any value would probably come from a smoothed / rolling version, not the instantaneous reading.
- **Wave direction (sin component):** r ≈ 0.17–0.19, small but non-zero. The cosine component is noise. Consistent with swell direction carrying weak regime information.
- **SST:** r ≈ 0.20 that *increases* slightly with horizon — that's a climatological / seasonal footprint, not a dynamical signal.
- **The lookback grid is monotonically decreasing in every column.** `now` is always the best single snapshot for predicting any future horizon; `3h ago`, `6h ago`, etc., are strictly worse. For `hsig_m` alone, extra history doesn't add information — it just restates the same signal, attenuated.

**What this means for modelling.**

1. We're in a regime where the only features that move the needle are `hsig_m` itself, `hmax_m` (with caveats), and `tz_s`. Direction and SST are at best 1–2% RMSE reductions each.
2. The flat lookback grid explains why our tuned sequence models didn't help: long context windows don't add information for this target. A short recent window plus maybe rolling statistics is all the signal there is.
3. At +48h the total self-persistence is already down to 0.46 (squared ≈ 21% R²). At +72h it's 0.31 (~10% R²). The physical ceiling for *this set of inputs* flattens hard past a day.
4. Real gains from here require new inputs, not new architectures — atmospheric forcing (e.g. BOM synoptic winds, GFS reanalysis) is the obvious candidate. The buoy sees the ocean's state; the wind field drives its evolution.


## 3. Build target and feature matrix

Target: `y.loc[t] = hsig_m.loc[t + 12h]`, produced by `make_target`.

Features: start from the raw channels, then add

- **sin/cos of wave direction** (circular variable — otherwise the 359\u00b0\u21921\u00b0 jump looks like a 358\u00b0 swing to any regressor)
- **time-of-day and day-of-year cyclical encodings**
- **lag features** at 30min, 1h, 2h, 4h, 12h, 24h — capture short-term dynamics and the diurnal cycle
- **rolling means and stds** over 3h, 12h, 24h — smooth state + volatility proxy

All rolling features are shifted by 1 step so they never peek at the current observation (enforced by a test in `test_forecast.py`).

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = fc.encode_circular(df)
    X = fc.add_time_features(X)
    X = fc.add_lag_features(
        X,
        columns=["hsig_m", "hmax_m", "tp_s", "tz_s", "sst_c"],
        lags=[1, 2, 4, 8, 24, 48],
    )
    X = fc.add_rolling_features(
        X,
        columns=["hsig_m", "tp_s"],
        windows=[6, 24, 48],
        stats=("mean", "std"),
    )
    return X

X_full = build_features(df)
y_full = fc.make_target(df)

print(f"X shape: {X_full.shape}  |  feature count: {X_full.shape[1]}")
print(f"y NaN count (should equal horizon steps = {fc.HORIZON_STEPS}): {y_full.isna().sum()}")

In [ ]:
X_train, X_test, y_train, y_test = fc.chronological_split(X_full, y_full, test_frac=0.2)
print(f"train: {len(X_train):,}  ({X_train.index.min().date()} \u2192 {X_train.index.max().date()})")
print(f"test:  {len(X_test):,}  ({X_test.index.min().date()} \u2192 {X_test.index.max().date()})")

## 4. Baselines

- **Persistence:** ŷ = hsig\_m(t). With autocorrelation >0.8 at 12h (plot above), this is a tough baseline to beat.
- **Seasonal naive (24h):** ŷ = hsig\_m(t − 12h). Uses the fact that conditions 24h ago at the *target* time are correlated with now.
- **Climatology (hour-of-day):** ŷ = training-set mean of hsig\_m at the forecast hour. The floor any real model must beat.

Skill score is computed relative to persistence (the dominant baseline).

In [ ]:
persistence = fc.evaluate(
    fc.PersistenceForecaster(), X_train, y_train, X_test, y_test, name="persistence"
)
baseline_preds = persistence.predictions  # reference for skill-score-vs-persistence

seasonal = fc.evaluate(
    fc.SeasonalNaiveForecaster(period_steps=48), X_train, y_train, X_test, y_test,
    name="seasonal-naive-24h", baseline_preds=baseline_preds,
)
climatology = fc.evaluate(
    fc.ClimatologyHourForecaster(), X_train, y_train, X_test, y_test,
    name="climatology-hod", baseline_preds=baseline_preds,
)

fc.compare([persistence, seasonal, climatology])

## 5. Linear models

With 47 features and ~150k training rows, plain OLS is well-posed but likely to overfit on collinear lag columns. Ridge is the sensible default.

In [ ]:
linreg = fc.evaluate(
    LinearRegression(), X_train, y_train, X_test, y_test,
    name="linear-regression", baseline_preds=baseline_preds,
)
ridge = fc.evaluate(
    Ridge(alpha=1.0), X_train, y_train, X_test, y_test,
    name="ridge", baseline_preds=baseline_preds,
)
fc.compare([persistence, linreg, ridge])

## 6. Tree ensembles

Random Forest and Gradient Boosting capture non-linearities and feature interactions that lag-based linear models can't. Modest tree/leaf sizes keep this runnable on a laptop — a serious pass would tune via `TimeSeriesSplit` cross-validation.

In [ ]:
rf = fc.evaluate(
    RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    X_train, y_train, X_test, y_test,
    name="random-forest", baseline_preds=baseline_preds,
)
gb = fc.evaluate(
    GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    X_train, y_train, X_test, y_test,
    name="gradient-boosting", baseline_preds=baseline_preds,
)
fc.compare([persistence, ridge, rf, gb])

## 7. Sequence models (RNN / GRU / LSTM / TCN)

The baselines, linear, and tree models all consume a hand-crafted feature matrix with lags and rolling stats. Sequence models take a different approach: feed the last `seq_len` raw feature rows and let the network learn the temporal structure itself. Four variants compared here:

- **Simple RNN** — vanilla tanh recurrence. Baseline sequence model; vanishing-gradient prone on long horizons.
- **GRU** — gated recurrent unit. Fewer parameters than LSTM; often matches it in skill at this scale.
- **LSTM** — the default sequence model. Explicit cell state helps carry information across the 48-step (24h) context.
- **TCN** — Temporal Convolutional Network. Dilated causal 1D convolutions; parallelisable and with explicit receptive-field control. A genuinely different architecture against the three RNNs.

All four share the same harness (`_TorchSeqForecaster`): feature and target standardisation, a `seq_len=48` sliding window, Adam + MSE, 5 epochs. They also use a different feature frame — just the raw channels + sin/cos wave direction — since the recurrent/conv layers handle history on their own.


In [ ]:
# Raw feature frame: no lag/rolling columns, just the six channels with
# peak direction sin/cos encoded. Same index as X_full, so baseline_preds
# computed on the test set still aligns row-for-row.
X_seq_full = fc.encode_circular(df)
Xs_train, Xs_test, ys_train, ys_test = fc.chronological_split(X_seq_full, y_full, test_frac=0.2)
print(f"sequence-model feature count: {X_seq_full.shape[1]}  |  train rows: {len(Xs_train):,}")


In [ ]:
# All four share the same sweep: seq_len=48 (24h context), hidden=32,
# 1 layer, 5 epochs. Enough to fit meaningfully without blowing an hour on CPU.
SEQ_KW = dict(seq_len=48, hidden=32, num_layers=1, epochs=5, batch_size=512, lr=1e-3)

rnn = fc.evaluate(
    fc.SimpleRNNForecaster(**SEQ_KW), Xs_train, ys_train, Xs_test, ys_test,
    name="simple-rnn", baseline_preds=baseline_preds,
)
gru = fc.evaluate(
    fc.GRUForecaster(**SEQ_KW), Xs_train, ys_train, Xs_test, ys_test,
    name="gru", baseline_preds=baseline_preds,
)
lstm = fc.evaluate(
    fc.LSTMForecaster(**SEQ_KW), Xs_train, ys_train, Xs_test, ys_test,
    name="lstm", baseline_preds=baseline_preds,
)
tcn = fc.evaluate(
    fc.TCNForecaster(**SEQ_KW), Xs_train, ys_train, Xs_test, ys_test,
    name="tcn", baseline_preds=baseline_preds,
)
fc.compare([persistence, rnn, gru, lstm, tcn])


## 8. Full comparison

In [ ]:
all_results = [persistence, seasonal, climatology, linreg, ridge, rf, gb, rnn, gru, lstm, tcn]
summary = fc.compare(all_results)
summary


In [ ]:
viz.rmse_bar(
    all_results, title=f"Test-set RMSE — {fc.HORIZON_HOURS}h hsig_m forecast",
)
plt.tight_layout()
plt.show()


## 9. Error analysis on the best model

Beyond a single RMSE number, we want to know *where* the model fails. Two useful views:

- Residuals across the test window — are there regimes (storms, calms) where we systematically under/over-predict?
- Error binned by observed hsig_m — does the model degrade for extreme swells, the events surfers care about most?

In [ ]:
best = min(all_results, key=lambda r: r.metrics["RMSE"])
print(f"Best model: {best.name}  |  RMSE = {best.metrics['RMSE']:.4f} m")

pred_series = pd.Series(best.predictions, index=X_test.index, name="pred")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
viz.residual_timeseries(y_test, pred_series, model_name=best.name, ax=axes[0])
_, grouped = viz.residual_by_bin(
    y_test, pred_series,
    bins=[0, 0.75, 1.25, 1.75, 2.5, 10.0],
    statistic="std",
    title="Residual std by observed hsig_m bin",
    ylabel="std of errors (m)",
    ax=axes[1],
)
plt.tight_layout()
plt.show()
grouped


## 10. Next steps

Ideas that fit the existing infrastructure without notebook rewrites:

- **Hyperparameter search with walk-forward CV** — wrap `evaluate()` in a loop over `sklearn.model_selection.TimeSeriesSplit` folds.
- **Exogenous inputs** — merge BOM wind/pressure fields or GFS reanalysis; at 12h+ horizons atmospheric forcing starts to matter more than local persistence.
- **Direct vs. recursive multi-step** — build a multi-output target (e.g. t+1..t+24) and compare direct (one model per horizon) against recursive (one-step model iterated). See where error accumulation dominates.
- **Gradient-boosted libs** — LightGBM / XGBoost often win on this kind of tabular time-series problem; add to `requirements.txt` and slot them into the same `evaluate()` harness.
- **Sequence models** — a small LSTM/Temporal Convolution on the recent window; only worth it if it beats Ridge by a meaningful margin.